In [8]:
from pyspark.sql import SparkSession
import findspark
import pyspark.sql.functions as F
import plotly.express as px
import os
from pyspark.sql.functions import countDistinct

In [2]:
findspark.init()

In [3]:
spark = SparkSession.builder \
    .appName("US Accidents Analysis - Local Laptop") \
    .master("local[*]") \
    .config("spark.driver.memory", "6g") \
    .config("spark.sql.shuffle.partitions", "16") \
    .config("spark.default.parallelism", "16") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.kryoserializer.buffer.max", "512m") \
    .getOrCreate()

spark

In [4]:
Data_Path = "Data/Full_Data"

In [5]:
df = spark.read \
    .option("mergeSchema", "false") \
    .parquet(Data_Path)

In [6]:
df = df.repartition(16)

In [7]:
df.printSchema()

root
 |-- ID: string (nullable = true)
 |-- Severity: integer (nullable = true)
 |-- Start_Time: timestamp (nullable = true)
 |-- End_Time: timestamp (nullable = true)
 |-- Start_Lat: double (nullable = true)
 |-- Start_Lng: double (nullable = true)
 |-- End_Lat: double (nullable = true)
 |-- End_Lng: double (nullable = true)
 |-- Distance(mi): double (nullable = true)
 |-- Description: string (nullable = true)
 |-- Street: string (nullable = true)
 |-- City: string (nullable = true)
 |-- County: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Zipcode: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Timezone: string (nullable = true)
 |-- Airport_Code: string (nullable = true)
 |-- Weather_Timestamp: timestamp (nullable = true)
 |-- Temperature(F): double (nullable = true)
 |-- Wind_Chill(F): double (nullable = true)
 |-- Humidity(%): double (nullable = true)
 |-- Pressure(in): double (nullable = true)
 |-- Visibility(mi): double (nullable = true

In [9]:
cols = ["Country", "Amenity", "Bump", "Crossing", "Give_Way", "Junction", "No_Exit", "Railway", "Roundabout", "Station", "Stop", "Traffic_Calming", "Traffic_Signal"]

df.select([
    countDistinct(c).alias(c) for c in cols
]).show()

+-------+-------+----+--------+--------+--------+-------+-------+----------+-------+----+---------------+--------------+
|Country|Amenity|Bump|Crossing|Give_Way|Junction|No_Exit|Railway|Roundabout|Station|Stop|Traffic_Calming|Traffic_Signal|
+-------+-------+----+--------+--------+--------+-------+-------+----------+-------+----+---------------+--------------+
|      1|      2|   2|       2|       2|       2|      2|      2|         2|      2|   2|              2|             2|
+-------+-------+----+--------+--------+--------+-------+-------+----------+-------+----+---------------+--------------+



In [11]:
for elem in cols:
    df.groupBy(elem, "Severity") \
    .count() \
    .orderBy(elem, "Severity") \
    .show()

+-------+--------+-------+
|Country|Severity|  count|
+-------+--------+-------+
|     US|       1|  93419|
|     US|       2|8689972|
|     US|       3|1454442|
|     US|       4| 335903|
+-------+--------+-------+

+-------+--------+-------+
|Amenity|Severity|  count|
+-------+--------+-------+
|  false|       1|  91494|
|  false|       2|8576118|
|  false|       3|1449315|
|  false|       4| 332485|
|   true|       1|   1925|
|   true|       2| 113854|
|   true|       3|   5127|
|   true|       4|   3418|
+-------+--------+-------+

+-----+--------+-------+
| Bump|Severity|  count|
+-----+--------+-------+
|false|       1|  93373|
|false|       2|8685860|
|false|       3|1454110|
|false|       4| 335858|
| true|       1|     46|
| true|       2|   4112|
| true|       3|    332|
| true|       4|     45|
+-----+--------+-------+

+--------+--------+-------+
|Crossing|Severity|  count|
+--------+--------+-------+
|   false|       1|  66088|
|   false|       2|7720540|
|   false|       

In [12]:
drop_cols = [
    "ID", "Source", "TMC", "Description",
    "End_Time", "End_Lat", "End_Lng",
    "Country", "Zipcode", "Number",
    "Airport_Code", "Weather_Timestamp",
    "Timezone"
]

df = df.drop(*[c for c in drop_cols if c in df.columns])

df.printSchema()

root
 |-- Severity: integer (nullable = true)
 |-- Start_Time: timestamp (nullable = true)
 |-- Start_Lat: double (nullable = true)
 |-- Start_Lng: double (nullable = true)
 |-- Distance(mi): double (nullable = true)
 |-- Street: string (nullable = true)
 |-- City: string (nullable = true)
 |-- County: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Temperature(F): double (nullable = true)
 |-- Wind_Chill(F): double (nullable = true)
 |-- Humidity(%): double (nullable = true)
 |-- Pressure(in): double (nullable = true)
 |-- Visibility(mi): double (nullable = true)
 |-- Wind_Direction: string (nullable = true)
 |-- Wind_Speed(mph): double (nullable = true)
 |-- Precipitation(in): double (nullable = true)
 |-- Weather_Condition: string (nullable = true)
 |-- Amenity: boolean (nullable = true)
 |-- Bump: boolean (nullable = true)
 |-- Crossing: boolean (nullable = true)
 |-- Give_Way: boolean (nullable = true)
 |-- Junction: boolean (nullable = true)
 |-- No_Exit: boole

In [13]:
total_rows = df.count()

missing_df = df.select([
    (
        F.count(F.when(F.col(c).isNull(), c)) / total_rows * 100
    ).alias(c)
    for c in df.columns
])

# Convert to vertical format for readability
missing_df = missing_df.toPandas().T.reset_index()
missing_df.columns = ["Column", "Missing_Percentage"]

# Sort descending
missing_df = missing_df.sort_values(by="Missing_Percentage", ascending=False)

missing_df

,Column,Missing_Percentage
16,Precipitation(in),26.036625
10,Wind_Chill(F),23.347112
15,Wind_Speed(mph),6.896115
14,Wind_Direction,2.354712
13,Visibility(mi),2.342067
11,Humidity(%),2.338209
17,Weather_Condition,2.308503
9,Temperature(F),2.204774
12,Pressure(in),1.890335
32,Civil_Twilight,0.246961


In [14]:
fig = px.bar(
    missing_df.head(20),
    x="Column",
    y="Missing_Percentage",
    title="Top 20 Columns by Missing Percentage"
)

fig.show()

In [15]:
numeric_cols = [
    "Temperature(F)", "Humidity(%)", "Pressure(in)",
    "Visibility(mi)", "Wind_Speed(mph)", "Precipitation(in)",'Wind_Chill(F)'
]

medians = df.select([
    F.expr(f'percentile_approx(`{c}`, 0.5)').alias(c)
    for c in numeric_cols if c in df.columns
]).collect()[0].asDict()

df = df.fillna(medians)

In [16]:
categorical_cols = [
    "Wind_Direction",
    "Weather_Condition",
    "Sunrise_Sunset",
    "Astronomical_Twilight",
    "Nautical_Twilight",
    "Civil_Twilight",
    "City",
    'Street'
]

mode_dict = {}

for col in categorical_cols:
    if col in df.columns:
        mode_value = df.groupBy(col) \
                       .count() \
                       .orderBy(F.desc("count")) \
                       .first()

        if mode_value is not None:
            mode_dict[col] = mode_value[0]

# Fill missing values
df = df.fillna(mode_dict)

print(mode_dict)

{'Wind_Direction': 'CALM', 'Weather_Condition': 'Fair', 'Sunrise_Sunset': 'Day', 'Astronomical_Twilight': 'Day', 'Nautical_Twilight': 'Day', 'Civil_Twilight': 'Day', 'City': 'Miami', 'Street': 'I-95 N'}


In [17]:
total_rows = df.count()

missing_df = df.select([
    (
        F.count(F.when(F.col(c).isNull(), c)) / total_rows * 100
    ).alias(c)
    for c in df.columns
])

# Convert to vertical format for readability
missing_df = missing_df.toPandas().T.reset_index()
missing_df.columns = ["Column", "Missing_Percentage"]

# Sort descending
missing_df = missing_df.sort_values(by="Missing_Percentage", ascending=False)

missing_df

,Column,Missing_Percentage
0,Severity,0.0
1,Start_Time,0.0
2,Start_Lat,0.0
3,Start_Lng,0.0
4,Distance(mi),0.0
5,Street,0.0
6,City,0.0
7,County,0.0
8,State,0.0
9,Temperature(F),0.0


In [18]:
bool_cols = [c for c, t in df.dtypes if t == "boolean"]

for col in bool_cols:
    df = df.withColumn(col, F.col(col).cast("int"))

In [19]:
df.printSchema()

root
 |-- Severity: integer (nullable = true)
 |-- Start_Time: timestamp (nullable = true)
 |-- Start_Lat: double (nullable = true)
 |-- Start_Lng: double (nullable = true)
 |-- Distance(mi): double (nullable = true)
 |-- Street: string (nullable = false)
 |-- City: string (nullable = false)
 |-- County: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Temperature(F): double (nullable = false)
 |-- Wind_Chill(F): double (nullable = false)
 |-- Humidity(%): double (nullable = false)
 |-- Pressure(in): double (nullable = false)
 |-- Visibility(mi): double (nullable = false)
 |-- Wind_Direction: string (nullable = false)
 |-- Wind_Speed(mph): double (nullable = false)
 |-- Precipitation(in): double (nullable = false)
 |-- Weather_Condition: string (nullable = false)
 |-- Amenity: integer (nullable = true)
 |-- Bump: integer (nullable = true)
 |-- Crossing: integer (nullable = true)
 |-- Give_Way: integer (nullable = true)
 |-- Junction: integer (nullable = true)
 |-- No_

In [20]:
total_rows = df.count()

unique_rows = df.dropDuplicates().count()

duplicate_rows = total_rows - unique_rows

print("Total Rows:", total_rows)
print("Unique Rows:", unique_rows)
print("Duplicate Rows:", duplicate_rows)

Total Rows: 10573736
Unique Rows: 8735017
Duplicate Rows: 1838719


In [21]:
df = df.dropDuplicates()

print("After removing duplicates:", df.count())

After removing duplicates: 8735017


In [22]:
output_path = "Data/PreProcessed_Data"

In [23]:
df.write \
  .mode("overwrite") \
  .parquet(output_path)

In [24]:
parquet_data = "Data/PreProcessed_Data"
for file in os.listdir(parquet_data):
    if file.endswith(".parquet"):
        print(file)

part-00000-9cfe47b0-f966-4441-8a35-d9c3902278c9-c000.snappy.parquet
part-00001-9cfe47b0-f966-4441-8a35-d9c3902278c9-c000.snappy.parquet
part-00002-9cfe47b0-f966-4441-8a35-d9c3902278c9-c000.snappy.parquet
part-00003-9cfe47b0-f966-4441-8a35-d9c3902278c9-c000.snappy.parquet
part-00004-9cfe47b0-f966-4441-8a35-d9c3902278c9-c000.snappy.parquet
part-00005-9cfe47b0-f966-4441-8a35-d9c3902278c9-c000.snappy.parquet
part-00006-9cfe47b0-f966-4441-8a35-d9c3902278c9-c000.snappy.parquet
part-00007-9cfe47b0-f966-4441-8a35-d9c3902278c9-c000.snappy.parquet
part-00008-9cfe47b0-f966-4441-8a35-d9c3902278c9-c000.snappy.parquet
part-00009-9cfe47b0-f966-4441-8a35-d9c3902278c9-c000.snappy.parquet
part-00010-9cfe47b0-f966-4441-8a35-d9c3902278c9-c000.snappy.parquet
part-00011-9cfe47b0-f966-4441-8a35-d9c3902278c9-c000.snappy.parquet
part-00012-9cfe47b0-f966-4441-8a35-d9c3902278c9-c000.snappy.parquet
part-00013-9cfe47b0-f966-4441-8a35-d9c3902278c9-c000.snappy.parquet
part-00014-9cfe47b0-f966-4441-8a35-d9c3902278c9-